# Train on SUSE AI Factory from a local Jupyter

Submits a small PyTorch training run to the cluster's Kubeflow Trainer, scheduled by KAI in your
project's queue. You sign in to **Rancher with an API token**; no kubeconfig is needed.

Before you start: an admin has enabled training on your namespace and granted you **AI Trainer**
(SUSE AI Factory > Workloads > Scheduling > Projects).

In [ ]:
%pip install -q "kubeflow==0.4.1" "aif-training @ git+https://github.com/dstanley/aif@prototype/kubeflow-trainer#subdirectory=sdk/aif-training"

## 1. Connect to Rancher
Set these in the environment before starting Jupyter, or type them when asked. The token is a Rancher API key (avatar > **Account & API Keys** > Create), scoped to no cluster or to this one.

In [ ]:
import os, getpass, ssl, urllib.request, tempfile

RANCHER_URL = os.environ.get("RANCHER_URL") or input("Rancher URL (e.g. https://rancher.example.com): ")
RANCHER_TOKEN = os.environ.get("RANCHER_TOKEN") or getpass.getpass("Rancher API token (token-xxxxx:yyyy): ")
CLUSTER_ID = os.environ.get("CLUSTER_ID", "local")        # Rancher's cluster ID (c-xxxxx for downstream)
NAMESPACE = os.environ.get("NAMESPACE", "aif-submit")     # your project namespace with training enabled
# Only when the URL's host is not a name on Rancher's certificate: the name to verify instead
TLS_SERVER_NAME = os.environ.get("RANCHER_TLS_SERVER_NAME") or None

In [ ]:
from kubernetes import client

# Rancher's own CA (public), so TLS is verified even with a private CA
ctx = ssl.create_default_context(); ctx.check_hostname = False; ctx.verify_mode = ssl.CERT_NONE
ca_pem = urllib.request.urlopen(f"{RANCHER_URL}/cacerts", context=ctx).read().decode()
ca_file = None
if "BEGIN CERTIFICATE" in ca_pem:                      # empty when Rancher uses a public CA
    ca_file = tempfile.NamedTemporaryFile("w", suffix=".pem", delete=False); ca_file.write(ca_pem); ca_file.close()

cfg = client.Configuration()
cfg.host = f"{RANCHER_URL}/k8s/clusters/{CLUSTER_ID}"   # Rancher's proxy: every call runs with your Rancher rights
cfg.api_key = {"authorization": f"Bearer {RANCHER_TOKEN}"}
if ca_file:
    cfg.ssl_ca_cert = ca_file.name
cfg.tls_server_name = TLS_SERVER_NAME

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
trainer = TrainerClient(backend_config=KubernetesBackendConfig(namespace=NAMESPACE, client_configuration=cfg))
for r in trainer.list_runtimes():
    print(r.name)

## 2. The training function
It runs on the cluster, not here: the SDK ships its source. Whatever it writes to `$OUTPUT_DIR` is kept on the run's volume.

In [ ]:
def train():
    import os, math, torch
    # torchrun starts one process per CPU slot or GPU (aif-cpu: 2); without DDP each trains alone,
    # so only rank 0 reports and saves
    rank0 = int(os.environ.get("RANK", "0")) == 0
    torch.manual_seed(0)
    # two interleaved spirals: a small non-linear classification problem
    n = 2000
    t = torch.rand(n) * 3 * math.pi
    cls = torch.randint(0, 2, (n,))
    r = t / (3 * math.pi)
    x = torch.stack([r * torch.cos(t + cls * math.pi), r * torch.sin(t + cls * math.pi)], 1) + 0.02 * torch.randn(n, 2)
    tr, te = slice(0, 1600), slice(1600, n)
    model = torch.nn.Sequential(torch.nn.Linear(2, 64), torch.nn.ReLU(), torch.nn.Linear(64, 64), torch.nn.ReLU(), torch.nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.01)
    for epoch in range(1, 301):
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(model(x[tr]), cls[tr])
        loss.backward(); opt.step()
        if rank0 and epoch % 50 == 0:
            acc = (model(x[te]).argmax(1) == cls[te]).float().mean().item()
            print(f"epoch {epoch}: loss {loss.item():.4f}, test accuracy {acc:.3f}", flush=True)
    out = os.environ.get("OUTPUT_DIR")
    if out and rank0:
        torch.save(model.state_dict(), os.path.join(out, "spiral-mlp.pt"))
        print("saved", os.path.join(out, "spiral-mlp.pt"))

## 3. Submit to your project's queue
`namespace_queue` puts the run in the KAI queue of your namespace (the admission policy refuses it otherwise). `aif-cpu` runs on CPUs; `aif-share-4g`/`aif-share-8g` on a GPU share; `aif-gpu` on whole GPUs.

In [ ]:
from kubeflow.trainer import CustomTrainer
from aif_training import namespace_queue

job = trainer.train(runtime=trainer.get_runtime("aif-cpu"),
                    trainer=CustomTrainer(func=train),
                    options=[namespace_queue(trainer)])
print("submitted", job)

## 4. Follow it
The run also shows in SUSE AI Factory > Workloads > Scheduling > Training Jobs.

In [ ]:
trainer.wait_for_job_status(job, timeout=1800)
print("\n".join(trainer.get_job_logs(job)))